# Setup — Ligações e Criação dos Schemas

Este notebook é o ponto de partida do pipeline ELT do Data Warehouse WWI.

**O que faz este notebook:**
1. Define os parâmetros de ligação às duas bases de dados
2. Testar as ligações (fonte operacional e DW)
3. Cria os schemas `bronze`, `silver` e `gold` na base de dados do DW
4. Cria as tabelas da camada `gold` (dimensões e tabela de factos)
5. Valida a estrutura criada

---
**Arquitectura Medalão**

```
wwi (fonte)                     wwi_dw (destino)
──────────────────────          ───────────────────────────────────────
wwi.*                   ──E─▶  bronze.*   (cópia raw)
                        ──T─▶  silver.*   (limpeza e normalização)
                        ──L─▶  gold.*     (esquema estrela / DW)
```

## 1. Instalação de dependências

Bibliotecas necessárias:
- `psycopg2-binary` — driver PostgreSQL
- `sqlalchemy` — abstração de ligação e execução de SQL
- `pandas` — manipulação de dados nos notebooks silver e gold
- `python-dotenv` — carregamento de credenciais a partir de ficheiro `.env`

In [33]:
%pip install psycopg2-binary sqlalchemy pandas python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


## 2. Parâmetros de ligação

As credenciais são carregadas a partir do ficheiro `.env` na mesma pasta.


In [34]:
from dotenv import load_dotenv
import os

load_dotenv() 

# -- Fonte operacional -------------------------------------------------------
SRC_HOST     = os.getenv("SRC_HOST")
SRC_PORT     = int(os.getenv("SRC_PORT", 5432))
SRC_DB       = os.getenv("SRC_DB")
SRC_USER     = os.getenv("SRC_USER")
SRC_PASSWORD = os.getenv("SRC_PASSWORD")

# -- Data Warehouse -----------------------------------------------
DWH_HOST     = os.getenv("DWH_HOST")
DWH_PORT     = int(os.getenv("DWH_PORT", 5432))
DWH_DB       = os.getenv("DWH_DB")
DWH_USER     = os.getenv("DWH_USER")
DWH_PASSWORD = os.getenv("DWH_PASSWORD")

# -- Schemas da arquitectura para medalhão -----------------------------------------
SCHEMAS = ["bronze", "silver", "gold"]

print(f"SRC: {SRC_USER}@{SRC_HOST}:{SRC_PORT}/{SRC_DB}")
print(f"DWH: {DWH_USER}@{DWH_HOST}:{DWH_PORT}/{DWH_DB}")

SRC: dss@postgres2.ipca.pt:5432/wwi
DWH: postgres@localhost:5432/wwi_dw


## 3. Criação dos engines SQLAlchemy

In [35]:
from sqlalchemy import create_engine, text

def make_engine(host, port, db, user, password):
    url = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}"
    return create_engine(url)

engine_src = make_engine(SRC_HOST, SRC_PORT, SRC_DB, SRC_USER, SRC_PASSWORD)
engine_dwh = make_engine(DWH_HOST, DWH_PORT, DWH_DB, DWH_USER, DWH_PASSWORD)

print("Engines criados com sucesso.")

Engines criados com sucesso.


## 4. Teste das ligações

Verifica se ambas as bases de dados estão acessíveis antes de prosseguir.

In [36]:
def test_connection(engine, label):
    try:
        with engine.connect() as conn:
            result = conn.execute(text("SELECT current_database(), version()"))
            db_name, version = result.fetchone()
            print(f"✓ {label}")
            print(f"  Base de dados : {db_name}")
            print(f"  Versão        : {version.split(',')[0]}")
    except Exception as e:
        print(f"✗ {label} — ERRO: {e}")
    print()

test_connection(engine_src, "Fonte operacional wwi")
test_connection(engine_dwh, "Data Warehouse   wwi_dw")

✓ Fonte operacional wwi
  Base de dados : wwi
  Versão        : PostgreSQL 17.9 on x86_64-pc-linux-gnu

✓ Data Warehouse   wwi_dw
  Base de dados : wwi_dw
  Versão        : PostgreSQL 18.3 on x86_64-windows



## 5. Criação dos schemas da arquitectura medalão

Os três schemas são criados na base de dados do DW. `IF NOT EXISTS` garante idempotência — este notebook pode ser executado múltiplas vezes sem erros.

In [37]:
with engine_dwh.begin() as conn:
    for schema in SCHEMAS:
        conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {schema}"))
        print(f"✓ Schema '{schema}' criado (ou já existia)")

print("\nSchemas da arquitectura medalão prontos.")

✓ Schema 'bronze' criado (ou já existia)
✓ Schema 'silver' criado (ou já existia)
✓ Schema 'gold' criado (ou já existia)

Schemas da arquitectura medalão prontos.


## 6. Criação das tabelas da camada Gold

A camada gold corresponde ao Data Mart analítico da WWI, organizado em esquema estrela segundo a metodologia Kimball

**Dimensões com SCD Tipo 2** (`dim_stockitem`):
Esta dimensão rastreia o histórico de alterações dos atributos dos produtos (como marcas ou categorias), garantindo a fidelidade das análises temporais:
- `version` — número da versão do registo
- `date_from` — data de início de validade
- `date_to` — data de fim de validade (`NULL` = registo atual)
- `is_current` — indicador booleano para identificar a versão ativa.

**Dimensões com SCD Tipo 1** (`dim_customer`, `dim_geography`, `dim_salesperson`):
Dimensões que mantêm apenas o estado atual dos dados, sobrescrevendo valores antigos para simplificar a estrutura conforme planeado.

**Dimensão calendário** (`dim_date`): Completamente desnormalizada e gerada programaticamente para cobrir o intervalo de 2017 a 2020, facilitando a análise de tendências temporais, gerada programaticamente no notebook `03_gold.ipynb`.

**Tabelas de factos** (`ft_sales`, `ft_orders`, `ft_customertransactions`): Granularidade definida ao nível da linha de transação para permitir o máximo detalhe analítico:
- `ft_sales` — baseada nas linhas de fatura (invoicelines), focada em métricas de lucro líquido (LineProfit)
- `ft_orders` — baseada nas linhas de encomenda (orderlines), usada para medir a procura real.
- `ft_customertransactions` — regista transações financeiras e saldos em dívida (OutstandingBalance)


In [38]:
DDL_GOLD = """
-- 1. Dimensões SCD Tipo 2 (Histórico)
CREATE TABLE IF NOT EXISTS gold.Dim_StockItem (
    StockItemKey      SERIAL PRIMARY KEY,
    version           INT NOT NULL,
    date_from         TIMESTAMP NOT NULL,
    date_to           TIMESTAMP,
    is_current        BOOLEAN DEFAULT TRUE,
    StockItemID       INT NOT NULL,
    StockItemName     VARCHAR(255),
    Brand             VARCHAR(255),
    ColorName         VARCHAR(255),
    Size              VARCHAR(255),
    LeadTimeDays      INT,
    QuantityPerOuter  INT,
    StockGroupName    VARCHAR(255)
);

-- 2. Dimensões SCD Tipo 1 (Atualização direta)
CREATE TABLE IF NOT EXISTS gold.Dim_Customer (
    CustomerKey           SERIAL PRIMARY KEY,
    CustomerID            INT NOT NULL,
    CustomerName          VARCHAR(255) NOT NULL,
    BuyingGroupName       VARCHAR(255),
    CustomerCategoryName  VARCHAR(255),
    CreditLimit           NUMERIC(18,2),
    AccountOpenedDate     DATE,
    PaymentDays           INT
);

CREATE TABLE IF NOT EXISTS gold.Dim_Salesperson (
    SalespersonKey  SERIAL PRIMARY KEY,
    SalespersonID   INT NOT NULL,
    FullName        VARCHAR(255) NOT NULL,
    PreferredName   VARCHAR(255),
    PhoneNumber     VARCHAR(255)
);

CREATE TABLE IF NOT EXISTS gold.Dim_Geography (
    GeographyKey             SERIAL PRIMARY KEY,
    CityID                   INT NOT NULL,
    CityName                 VARCHAR(255) NOT NULL,
    StateProvinceName        VARCHAR(255),
    CountryName              VARCHAR(255),
    LatestRecordedPopulation BIGINT
);

CREATE TABLE IF NOT EXISTS gold.Dim_Date (
    DateKey     INT PRIMARY KEY,
    FullDate    DATE NOT NULL,
    Year        INT NOT NULL,
    Quarter     INT NOT NULL,
    MonthName   VARCHAR(255) NOT NULL,
    Month       INT NOT NULL
);

-- 3. Tabelas de Factos
CREATE TABLE IF NOT EXISTS gold.Fact_Sales (
    InvoiceLineID   INT NOT NULL, -- Dimensão Degenerada
    DateKey         INT NOT NULL REFERENCES gold.Dim_Date(DateKey),
    CustomerKey     INT NOT NULL REFERENCES gold.Dim_Customer(CustomerKey),
    StockItemKey    INT NOT NULL REFERENCES gold.Dim_StockItem(StockItemKey),
    GeographyKey    INT NOT NULL REFERENCES gold.Dim_Geography(GeographyKey),
    SalespersonKey  INT NOT NULL REFERENCES gold.Dim_Salesperson(SalespersonKey),
    Quantity        INT NOT NULL,
    ExtendedPrice   NUMERIC(18,2) NOT NULL,
    LineProfit      NUMERIC(18,2) NOT NULL,
    TaxAmount       NUMERIC(18,2) NOT NULL
);

CREATE TABLE IF NOT EXISTS gold.Fact_Orders (
    OrderLineID     INT NOT NULL, -- Dimensão Degenerada
    DateKey         INT NOT NULL REFERENCES gold.Dim_Date(DateKey),
    CustomerKey     INT NOT NULL REFERENCES gold.Dim_Customer(CustomerKey),
    StockItemKey    INT NOT NULL REFERENCES gold.Dim_StockItem(StockItemKey),
    GeographyKey    INT NOT NULL REFERENCES gold.Dim_Geography(GeographyKey),
    SalespersonKey  INT NOT NULL REFERENCES gold.Dim_Salesperson(SalespersonKey),
    Quantity        INT NOT NULL,
    UnitPrice       NUMERIC(18,2) NOT NULL,
    TaxRate         NUMERIC(18,2) NOT NULL
);

CREATE TABLE IF NOT EXISTS gold.Fact_CustomerTransactions (
    CustomerTransactionID INT NOT NULL, -- Dimensão Degenerada
    DateKey               INT NOT NULL REFERENCES gold.Dim_Date(DateKey),
    CustomerKey           INT NOT NULL REFERENCES gold.Dim_Customer(CustomerKey),
    TransactionAmount     NUMERIC(18,2) NOT NULL,
    OutstandingBalance    NUMERIC(18,2) NOT NULL,
    IsFinalized           SMALLINT NOT NULL
);
"""

with engine_dwh.begin() as conn:
    conn.execute(text(DDL_GOLD))

print("✓ Tabelas da camada gold (WWI Data Mart) criadas com sucesso.")

✓ Tabelas da camada gold (WWI Data Mart) criadas com sucesso.


## 7. Validação — inventário dos schemas e tabelas criadas

In [39]:
import pandas as pd
from sqlalchemy import text

query = """
    SELECT
        table_schema  AS schema,
        table_name    AS tabela,
        table_type    AS tipo
    FROM information_schema.tables
    WHERE table_schema IN ('bronze', 'silver', 'gold')
    ORDER BY table_schema, table_name
"""

with engine_dwh.connect() as conn:
    df = pd.read_sql(text(query), conn)

print("Inventário de Objectos no Data Mart WWI:")
if df.empty:
    print("Atenção: Não foram encontrados objetos. Verifica se o DDL foi executado.")
else:
    print(df.to_string(index=False))

Inventário de Objectos no Data Mart WWI:
schema                    tabela       tipo
bronze             _load_control BASE TABLE
bronze              buyinggroups BASE TABLE
bronze                    cities BASE TABLE
bronze                    colors BASE TABLE
bronze                 countries BASE TABLE
bronze        customercategories BASE TABLE
bronze                 customers BASE TABLE
bronze      customertransactions BASE TABLE
bronze              invoicelines BASE TABLE
bronze                  invoices BASE TABLE
bronze                orderlines BASE TABLE
bronze                    orders BASE TABLE
bronze                    people BASE TABLE
bronze            stateprovinces BASE TABLE
bronze               stockgroups BASE TABLE
bronze                stockitems BASE TABLE
bronze      stockitemstockgroups BASE TABLE
  gold              dim_customer BASE TABLE
  gold                  dim_date BASE TABLE
  gold             dim_geography BASE TABLE
  gold           dim_salesperson BA

## 8. Validação — tabelas da fonte operacional

In [40]:
query_src = """
    SELECT
        table_name AS tabela,
        (SELECT COUNT(*)
         FROM information_schema.columns c
         WHERE c.table_name = t.table_name
           AND c.table_schema = 'public') AS colunas
    FROM information_schema.tables t
    WHERE table_schema = 'public'
      AND table_type = 'BASE TABLE'
    ORDER BY table_name
"""

with engine_src.connect() as conn:
    df_src = pd.read_sql(text(query_src), conn)

print(f"Tabelas detetadas na fonte operacional ({SRC_DB}):")
if df_src.empty:
    print("Nenhuma tabela encontrada no schema 'public'. Verifica a ligação ou o nome da DB.")
else:
    print(df_src.to_string(index=False))

Tabelas detetadas na fonte operacional (wwi):
              tabela  colunas
        buyinggroups        2
              cities        5
              colors        2
           countries        8
  customercategories        2
           customers       28
customertransactions       12
     deliverymethods        2
        invoicelines       11
            invoices       23
          orderlines        9
              orders       14
        packagetypes        2
      paymentmethods        2
              people       18
        specialdeals       12
      stateprovinces        6
         stockgroups        2
          stockitems       22
    transactiontypes        2


## 9. Resumo

Se as células anteriores correram sem erros, o ambiente para o projeto WWI_DW está oficialmente preparado:

| Passo | Descrição | Estado |
|---|---|---|
| Ligação à fonte operacional | Conexão estabelecida com o servidor do IPCA (wwi) | ✓ |
| Ligação ao DW | Conexão estabelecida com o PostgreSQL local (wwi_dw) | ✓ |
| Arquitetura Medalhão | Schemas bronze, silver e gold garantidos. | ✓ |
| Data Mart (Gold) |Tabelas de Factos e Dimensões (incluindo SCD2) criadas. | ✓ |
| SCD Tipo 2 | Colunas de histórico configuradas na Dim_StockItem. | ✓ |

**Próximo passo:** executar o notebook `01_bronze.ipynb`.